# FlashNorm M3 V6: TB_K=32 + 3-stage pipeline decode kernel

## Why V5 lost at Llama decode

V5's per-K-step time was ~0.55 us (~770 cycles) across all shapes. That is K-invariant fixed overhead from cp.async.commit_group + wait_group + 2 __syncthreads + WMMA. Total time = K-steps * per-step. SmolLM2 K=576 has 36 steps; Llama-8B K=4096 has 256 steps. Same per-step, but 7x more steps -> K dominates.

cuBLAS at Llama-8B M=1 runs at ~0.23 us per K-step because it uses warp-specialized pipelines that overlap load and compute more aggressively than our uniform pattern.

## V6 fixes

Two compounding changes:

1. **TB_K = 32** (doubled from 16). Halves K-step count.
2. **3 pipeline stages** (up from 2). With 3 stages, when we wait for stage N's data, stages N+1 and N+2 are already in flight, so HBM latency is fully hidden even at short iteration bodies.

Target at Llama-8B M=1: 128 steps * ~0.30 us = 38 us vs realistic 58 us -> **WIN +34%**

## What this does NOT claim to fix

- Matmul quality at large-M Llama prefill (still V1)
- Warp-specialization gap vs cuBLAS (we still use uniform warps)

If V6 lands 6 decode wins and preserves SmolLM2 wins, we have the paper's end-to-end result.

## 1. Setup

In [ ]:
import subprocess, os
subprocess.run(['pip', 'install', '-q', 'ninja'], check=True)
import torch
assert 'A100' in torch.cuda.get_device_name(0)
os.environ['TORCH_CUDA_ARCH_LIST'] = '8.0'
print('ok')

## 2. Shapes + baselines

In [ ]:
SHAPES = {
    'SmolLM2-135M':  (576, 960),
    'Llama-3.2-1B':  (2048, 2560),
    'Llama-3.1-8B':  (4096, 6144),
}
M_VALUES = [1, 16, 64, 256, 1024, 4096]
EPS = 1e-6
BENCH_ITERS = 100
WARMUP_ITERS = 20

V5_TIMES_MS = {
    ('SmolLM2-135M', 1): 0.0235, ('SmolLM2-135M', 16): 0.0239,
    ('SmolLM2-135M', 64): 0.0297, ('SmolLM2-135M', 256): 0.0301,
    ('SmolLM2-135M', 1024): 0.0433, ('SmolLM2-135M', 4096): 0.1109,
    ('Llama-3.2-1B', 1): 0.0735, ('Llama-3.2-1B', 16): 0.0729,
    ('Llama-3.2-1B', 64): 0.0868, ('Llama-3.2-1B', 256): 0.1139,
    ('Llama-3.2-1B', 1024): 0.2709, ('Llama-3.2-1B', 4096): 0.6585,
    ('Llama-3.1-8B', 1): 0.1318, ('Llama-3.1-8B', 16): 0.1339,
    ('Llama-3.1-8B', 64): 0.1507, ('Llama-3.1-8B', 256): 0.2220,
    ('Llama-3.1-8B', 1024): 0.8148, ('Llama-3.1-8B', 4096): 3.1533,
}

def bench(fn, *args, iters=BENCH_ITERS, warmup=WARMUP_ITERS):
    for _ in range(warmup): fn(*args)
    torch.cuda.synchronize()
    s = torch.cuda.Event(enable_timing=True); e = torch.cuda.Event(enable_timing=True)
    s.record()
    for _ in range(iters): fn(*args)
    e.record()
    torch.cuda.synchronize()
    return s.elapsed_time(e) / iters

HAS_NATIVE_RMS = hasattr(torch.nn.functional, 'rms_norm')
def realistic_rms_matmul(x, W, eps=EPS):
    weight = torch.ones(x.shape[-1], dtype=x.dtype, device=x.device)
    if HAS_NATIVE_RMS:
        x_norm = torch.nn.functional.rms_norm(x, (x.shape[-1],), weight, eps)
    else:
        var = x.pow(2).mean(-1, keepdim=True).to(torch.float32)
        inv = torch.rsqrt(var + eps).to(x.dtype)
        x_norm = x * inv
    return x_norm @ W

def ref_flashnorm(x, W, eps=EPS):
    x_f32 = x.to(torch.float32)
    inv_rms = torch.rsqrt(x_f32.pow(2).mean(-1, keepdim=True) + eps)
    return (x_f32 * inv_rms).to(x.dtype) @ W

print('ok')

## 3. V6 decode kernel: TB_K=32 + 3 stages

In [ ]:
DECODE_SRC = r"""
#include <cuda_runtime.h>
#include <cuda_fp16.h>
#include <mma.h>
#include <torch/extension.h>
using namespace nvcuda;

static constexpr int TB_M_DEC = 16;
static constexpr int TB_N_DEC = 64;
static constexpr int TB_K_DEC = 32;           // doubled from V5
static constexpr int NUM_STAGES_DEC = 3;      // up from 2
static constexpr int WM_D = 16, WN_D = 16, WK_D = 16;
static constexpr int NUM_THREADS_DEC = 128;

__device__ __forceinline__ unsigned smem_addr_d(const void* p) {
    return static_cast<unsigned>(__cvta_generic_to_shared(p));
}
__device__ __forceinline__ void cp_async_16B_d(void* dst, const void* src) {
    unsigned s = smem_addr_d(dst);
    asm volatile("cp.async.cg.shared.global [%0], [%1], 16;\n" :: "r"(s), "l"(src));
}

__global__ void flashnorm_decode_kernel(
    const __half* __restrict__ x,
    const __half* __restrict__ W,
    __half* __restrict__ out,
    int M, int N, int K, float eps
) {
    int block_n = blockIdx.x * TB_N_DEC;
    int tid = threadIdx.x;
    int warp_id = tid / 32;
    int warp_n_base = warp_id * WN_D;

    __shared__ __align__(16) __half smem_A[NUM_STAGES_DEC][TB_M_DEC][TB_K_DEC + 8];
    __shared__ __align__(16) __half smem_B[NUM_STAGES_DEC][TB_K_DEC][TB_N_DEC + 8];
    __shared__ float smem_acc[TB_M_DEC][TB_N_DEC];
    __shared__ float smem_rms[TB_M_DEC];

    // 2 WMMA-K slices per iter (TB_K=32 / WK=16)
    wmma::fragment<wmma::matrix_a, WM_D, WN_D, WK_D, __half, wmma::row_major> a_frag[2];
    wmma::fragment<wmma::matrix_b, WM_D, WN_D, WK_D, __half, wmma::row_major> b_frag[2];
    wmma::fragment<wmma::accumulator, WM_D, WN_D, WK_D, float> c_frag;
    wmma::fill_fragment(c_frag, 0.0f);

    float thread_rms_sq = 0.0f;
    int my_row = tid;
    bool row_valid = (my_row < TB_M_DEC) && (my_row < M);

    int num_iters = (K + TB_K_DEC - 1) / TB_K_DEC;

    // A tile: 16 * 32 = 512 halfs = 64 chunks of 8. Use 64 threads.
    // B tile: 32 * 64 = 2048 halfs = 256 chunks of 8. 128 threads * 2 chunks.
    int a_chunk_idx = tid;                   // 0..127 (first 64 are valid)
    int a_chunk_r = tid / 4;                 // 0..15 (when tid < 64)
    int a_chunk_col = (tid % 4) * 8;         // 0, 8, 16, 24

    int b0_r = tid / 8;                      // 0..15 (first half of B tile, k=0..15)
    int b0_col = (tid % 8) * 8;
    int b1_r = 16 + (tid / 8);               // 16..31 (second half, k=16..31)
    int b1_col = (tid % 8) * 8;

    auto issue_async_loads = [&](int stage, int k_idx) {
        int k0 = k_idx * TB_K_DEC;

        // A load (64 threads active)
        if (tid < 64) {
            int m_g = a_chunk_r;
            int k_g = k0 + a_chunk_col;
            bool inA = (m_g < M) && (k_g + 7 < K);
            __half* sa = &smem_A[stage][a_chunk_r][a_chunk_col];
            if (inA) cp_async_16B_d(sa, &x[m_g * K + k_g]);
            else *reinterpret_cast<uint4*>(sa) = make_uint4(0, 0, 0, 0);
        }

        // B load, first chunk (k=0..15)
        {
            int k_g = k0 + b0_r;
            int n_g = block_n + b0_col;
            bool inB = (k_g < K) && (n_g + 7 < N);
            __half* sb = &smem_B[stage][b0_r][b0_col];
            if (inB) cp_async_16B_d(sb, &W[k_g * N + n_g]);
            else *reinterpret_cast<uint4*>(sb) = make_uint4(0, 0, 0, 0);
        }

        // B load, second chunk (k=16..31)
        {
            int k_g = k0 + b1_r;
            int n_g = block_n + b1_col;
            bool inB = (k_g < K) && (n_g + 7 < N);
            __half* sb = &smem_B[stage][b1_r][b1_col];
            if (inB) cp_async_16B_d(sb, &W[k_g * N + n_g]);
            else *reinterpret_cast<uint4*>(sb) = make_uint4(0, 0, 0, 0);
        }
    };

    // Prefetch stages 0 and 1 (2 groups committed)
    if (num_iters > 0) {
        issue_async_loads(0, 0);
        asm volatile("cp.async.commit_group;\n");
    }
    if (num_iters > 1) {
        issue_async_loads(1, 1);
        asm volatile("cp.async.commit_group;\n");
    }

    // Mainloop with 3-stage pipeline
    for (int it = 0; it < num_iters; ++it) {
        int current = it % NUM_STAGES_DEC;

        // Prefetch it+2 (if exists) into stage (it+2)%3
        if (it + 2 < num_iters) {
            int prefetch_stage = (it + 2) % NUM_STAGES_DEC;
            issue_async_loads(prefetch_stage, it + 2);
            asm volatile("cp.async.commit_group;\n");
        } else {
            asm volatile("cp.async.commit_group;\n");  // empty group for consistent waiting
        }

        // Wait for current stage to be ready (at most 2 groups in flight)
        asm volatile("cp.async.wait_group 2;\n");
        __syncthreads();

        // RMS accumulation from current stage
        if (row_valid) {
            #pragma unroll
            for (int c = 0; c < TB_K_DEC; ++c) {
                int kg = it * TB_K_DEC + c;
                if (kg < K) {
                    float v = __half2float(smem_A[current][my_row][c]);
                    thread_rms_sq += v * v;
                }
            }
        }

        // WMMA: 2 K-slices (TB_K=32 = 2 * WK=16)
        #pragma unroll
        for (int ks = 0; ks < 2; ++ks) {
            wmma::load_matrix_sync(a_frag[ks], &smem_A[current][0][ks * WK_D], TB_K_DEC + 8);
            wmma::load_matrix_sync(b_frag[ks], &smem_B[current][ks * WK_D][warp_n_base], TB_N_DEC + 8);
            wmma::mma_sync(c_frag, a_frag[ks], b_frag[ks], c_frag);
        }

        __syncthreads();
    }

    asm volatile("cp.async.wait_all;\n");
    __syncthreads();

    // Epilogue
    if (tid < TB_M_DEC) smem_rms[tid] = thread_rms_sq;
    __syncthreads();
    if (tid < TB_M_DEC) {
        float mean_sq = smem_rms[tid] / (float)K;
        smem_rms[tid] = rsqrtf(mean_sq + eps);
    }
    __syncthreads();

    wmma::store_matrix_sync(&smem_acc[0][warp_n_base], c_frag, TB_N_DEC, wmma::mem_row_major);
    __syncthreads();

    for (int i = 0; i < 8; ++i) {
        int idx = i * NUM_THREADS_DEC + tid;
        int r = idx / TB_N_DEC;
        int c = idx % TB_N_DEC;
        if (r < M && r < TB_M_DEC) {
            int n_g = block_n + c;
            if (n_g < N) {
                float scaled = smem_acc[r][c] * smem_rms[r];
                out[r * N + n_g] = __float2half(scaled);
            }
        }
    }
}

torch::Tensor flashnorm_decode(torch::Tensor x, torch::Tensor W, double eps) {
    TORCH_CHECK(x.is_cuda() && W.is_cuda(), "CUDA");
    TORCH_CHECK(x.dtype() == torch::kHalf && W.dtype() == torch::kHalf, "fp16");
    TORCH_CHECK(x.is_contiguous() && W.is_contiguous(), "contiguous");
    int M = x.size(0), K = x.size(1), N = W.size(1);
    TORCH_CHECK(M <= TB_M_DEC, "decode requires M <= 16");
    TORCH_CHECK(K % TB_K_DEC == 0 || K >= TB_K_DEC, "K must be >= TB_K_DEC=32");

    auto out = torch::empty({M, N}, x.options());
    dim3 grid((N + TB_N_DEC - 1) / TB_N_DEC);
    dim3 block(NUM_THREADS_DEC);
    flashnorm_decode_kernel<<<grid, block>>>(
        reinterpret_cast<const __half*>(x.data_ptr<at::Half>()),
        reinterpret_cast<const __half*>(W.data_ptr<at::Half>()),
        reinterpret_cast<__half*>(out.data_ptr<at::Half>()),
        M, N, K, (float)eps);
    return out;
}
"""

## 4. Prefill kernel (V1 unchanged)

In [ ]:
V1_SRC = r"""
#include <cuda_runtime.h>
#include <cuda_fp16.h>
#include <mma.h>
#include <torch/extension.h>
using namespace nvcuda;

static constexpr int TB_M_P = 64;
static constexpr int TB_N_P = 64;
static constexpr int TB_K_P = 16;
static constexpr int NUM_STAGES_P = 2;
static constexpr int WM = 16, WN = 16, WK = 16;
static constexpr int NUM_THREADS_P = 128;

__device__ __forceinline__ unsigned smem_addr_p(const void* p) {
    return static_cast<unsigned>(__cvta_generic_to_shared(p));
}
__device__ __forceinline__ void cp_async_16B_p(void* dst, const void* src) {
    unsigned s = smem_addr_p(dst);
    asm volatile("cp.async.cg.shared.global [%0], [%1], 16;\n" :: "r"(s), "l"(src));
}

__global__ void flashnorm_prefill_kernel(
    const __half* __restrict__ A, const __half* __restrict__ B,
    __half* __restrict__ out, int M, int N, int K, float eps
) {
    int block_m = blockIdx.y * TB_M_P;
    int block_n = blockIdx.x * TB_N_P;
    int tid = threadIdx.x;
    int warp_id = tid / 32;
    int warp_row = warp_id / 2;
    int warp_col = warp_id % 2;
    int warp_m_base = warp_row * 32;
    int warp_n_base = warp_col * 32;

    __shared__ __align__(16) __half smem_A[NUM_STAGES_P][TB_M_P][TB_K_P + 8];
    __shared__ __align__(16) __half smem_B[NUM_STAGES_P][TB_K_P][TB_N_P + 8];
    __shared__ float smem_acc[TB_M_P][TB_N_P];
    __shared__ float smem_rms[TB_M_P];

    wmma::fragment<wmma::matrix_a, WM, WN, WK, __half, wmma::row_major> a_frag[2];
    wmma::fragment<wmma::matrix_b, WM, WN, WK, __half, wmma::row_major> b_frag[2];
    wmma::fragment<wmma::accumulator, WM, WN, WK, float> c_frag[2][2];
    #pragma unroll
    for (int i = 0; i < 2; ++i) {
        #pragma unroll
        for (int j = 0; j < 2; ++j) wmma::fill_fragment(c_frag[i][j], 0.0f);
    }

    float thread_rms_sq = 0.0f;
    int my_row = tid;
    bool row_valid = (my_row < TB_M_P) && ((block_m + my_row) < M);
    int num_iters = (K + TB_K_P - 1) / TB_K_P;

    int a_chunk_r = tid / 2;
    int a_chunk_col = (tid % 2) * 8;
    int b_chunk_r = tid / 8;
    int b_chunk_col = (tid % 8) * 8;

    auto load_stage = [&](int stage, int k_idx) {
        int k0 = k_idx * TB_K_P;
        {
            int m_g = block_m + a_chunk_r;
            int k_g = k0 + a_chunk_col;
            bool inA = (m_g < M) && (a_chunk_r < TB_M_P) && (k_g + 7 < K);
            __half* sa = &smem_A[stage][a_chunk_r][a_chunk_col];
            if (inA) cp_async_16B_p(sa, &A[m_g * K + k_g]);
            else *reinterpret_cast<uint4*>(sa) = make_uint4(0, 0, 0, 0);
        }
        {
            int k_g = k0 + b_chunk_r;
            int n_g = block_n + b_chunk_col;
            bool inB = (k_g < K) && (n_g + 7 < N);
            __half* sb = &smem_B[stage][b_chunk_r][b_chunk_col];
            if (inB) cp_async_16B_p(sb, &B[k_g * N + n_g]);
            else *reinterpret_cast<uint4*>(sb) = make_uint4(0, 0, 0, 0);
        }
    };

    if (num_iters > 0) { load_stage(0, 0); asm volatile("cp.async.commit_group;\n"); }
    for (int it = 0; it < num_iters; ++it) {
        int current = it % NUM_STAGES_P;
        int next_s = (it + 1) % NUM_STAGES_P;
        if (it + 1 < num_iters) { load_stage(next_s, it + 1); asm volatile("cp.async.commit_group;\n"); }
        else asm volatile("cp.async.commit_group;\n");
        asm volatile("cp.async.wait_group 1;\n");
        __syncthreads();
        if (row_valid) {
            #pragma unroll
            for (int c = 0; c < TB_K_P; ++c) {
                int kg = it * TB_K_P + c;
                if (kg < K) {
                    float v = __half2float(smem_A[current][my_row][c]);
                    thread_rms_sq += v * v;
                }
            }
        }
        #pragma unroll
        for (int i = 0; i < 2; ++i)
            wmma::load_matrix_sync(a_frag[i], &smem_A[current][warp_m_base + i * WM][0], TB_K_P + 8);
        #pragma unroll
        for (int j = 0; j < 2; ++j)
            wmma::load_matrix_sync(b_frag[j], &smem_B[current][0][warp_n_base + j * WN], TB_N_P + 8);
        #pragma unroll
        for (int i = 0; i < 2; ++i)
            #pragma unroll
            for (int j = 0; j < 2; ++j)
                wmma::mma_sync(c_frag[i][j], a_frag[i], b_frag[j], c_frag[i][j]);
        __syncthreads();
    }
    asm volatile("cp.async.wait_all;\n");
    __syncthreads();
    if (tid < TB_M_P) smem_rms[tid] = thread_rms_sq;
    __syncthreads();
    if (tid < TB_M_P) {
        float mean_sq = smem_rms[tid] / (float)K;
        smem_rms[tid] = rsqrtf(mean_sq + eps);
    }
    __syncthreads();
    #pragma unroll
    for (int i = 0; i < 2; ++i) {
        #pragma unroll
        for (int j = 0; j < 2; ++j)
            wmma::store_matrix_sync(&smem_acc[warp_m_base + i * WM][warp_n_base + j * WN],
                                     c_frag[i][j], TB_N_P, wmma::mem_row_major);
    }
    __syncthreads();
    for (int i = 0; i < 32; ++i) {
        int idx = i * NUM_THREADS_P + tid;
        int r = idx / TB_N_P;
        int c = idx % TB_N_P;
        if (r < TB_M_P) {
            int mg = block_m + r;
            int ng = block_n + c;
            if (mg < M && ng < N) {
                float scaled = smem_acc[r][c] * smem_rms[r];
                out[mg * N + ng] = __float2half(scaled);
            }
        }
    }
}

torch::Tensor flashnorm_prefill(torch::Tensor x, torch::Tensor W, double eps) {
    int M = x.size(0), K = x.size(1), N = W.size(1);
    auto out = torch::empty({M, N}, x.options());
    dim3 grid((N + TB_N_P - 1) / TB_N_P, (M + TB_M_P - 1) / TB_M_P);
    dim3 block(NUM_THREADS_P);
    flashnorm_prefill_kernel<<<grid, block>>>(
        reinterpret_cast<const __half*>(x.data_ptr<at::Half>()),
        reinterpret_cast<const __half*>(W.data_ptr<at::Half>()),
        reinterpret_cast<__half*>(out.data_ptr<at::Half>()),
        M, N, K, (float)eps);
    return out;
}
"""

## 5. Compile

In [ ]:
from torch.utils.cpp_extension import load_inline
import time, subprocess, glob

def dump_diag(tag):
    for d in glob.glob(f'/root/.cache/torch_extensions/py*_cu*/{tag}'):
        for f in sorted(os.listdir(d)):
            print(f'  {f}')
        if os.path.isfile(os.path.join(d, 'build.ninja')):
            print(open(os.path.join(d, 'build.ninja')).read())
        r = subprocess.run(['ninja', '-v'], cwd=d, capture_output=True, text=True)
        print(r.stdout[-4000:]); print(r.stderr[-6000:])

DECODE_CPP = '''
#include <torch/extension.h>
torch::Tensor flashnorm_decode(torch::Tensor x, torch::Tensor W, double eps);
'''
t0 = time.time()
try:
    decode_mod = load_inline(
        name='flashnorm_v6_decode', cpp_sources=DECODE_CPP, cuda_sources=DECODE_SRC,
        functions=['flashnorm_decode'],
        extra_cuda_cflags=['-O3', '-std=c++17', '-gencode=arch=compute_80,code=sm_80',
                           '--use_fast_math', '--expt-extended-lambda', '--ptxas-options=-v'],
        verbose=True)
    print(f'decode ok in {time.time()-t0:.1f}s')
except Exception as e:
    print(f'decode failed: {e}'); dump_diag('flashnorm_v6_decode'); raise

PREFILL_CPP = '''
#include <torch/extension.h>
torch::Tensor flashnorm_prefill(torch::Tensor x, torch::Tensor W, double eps);
'''
t0 = time.time()
try:
    prefill_mod = load_inline(
        name='flashnorm_v6_prefill', cpp_sources=PREFILL_CPP, cuda_sources=V1_SRC,
        functions=['flashnorm_prefill'],
        extra_cuda_cflags=['-O3', '-std=c++17', '-gencode=arch=compute_80,code=sm_80',
                           '--use_fast_math', '--expt-extended-lambda', '--ptxas-options=-v'],
        verbose=True)
    print(f'prefill ok in {time.time()-t0:.1f}s')
except Exception as e:
    print(f'prefill failed: {e}'); dump_diag('flashnorm_v6_prefill'); raise

## 6. Dispatch

In [ ]:
def flashnorm_v6(x, W, eps=EPS):
    if x.shape[0] <= 16:
        return decode_mod.flashnorm_decode(x, W, eps)
    return prefill_mod.flashnorm_prefill(x, W, eps)

print('dispatch: M<=16 -> decode (TB_K=32, 3 stages), else prefill (V1)')

## 7. Correctness

In [ ]:
def check_correct(M, K, N, tol=0.05):
    torch.manual_seed(0)
    x = torch.randn(M, K, dtype=torch.float16, device='cuda')
    W = torch.randn(K, N, dtype=torch.float16, device='cuda') * 0.02
    ref = ref_flashnorm(x, W)
    got = flashnorm_v6(x, W)
    max_abs = (ref.float() - got.float()).abs().max().item()
    cos = torch.nn.functional.cosine_similarity(
        ref.float().flatten(), got.float().flatten(), dim=0).item()
    path = 'decode' if M <= 16 else 'prefill'
    ok = max_abs < tol
    print(f'  M={M:>5} K={K:>5} N={N:>5} [{path:>7}]: max_abs={max_abs:.4f} cos={cos:.6f}  '
          f'{"PASS" if ok else "FAIL"}')
    del x, W
    return ok

print('=== correctness ===')
all_pass = True
for name, (K, N) in SHAPES.items():
    print(name)
    for M in M_VALUES:
        all_pass &= check_correct(M, K, N)
assert all_pass

## 8. Benchmark

In [ ]:
print(f'{"Model":<15} {"M":>6} {"Path":>8} {"realistic":>11} {"V5":>10} {"V6":>10} {"V6 vs real":>12} {"V6 vs V5":>10}')
print('-' * 86)

results = {}
for name, (K, N) in SHAPES.items():
    for M in M_VALUES:
        torch.manual_seed(0)
        x = torch.randn(M, K, dtype=torch.float16, device='cuda')
        W = torch.randn(K, N, dtype=torch.float16, device='cuda') * 0.02
        t_real = bench(realistic_rms_matmul, x, W)
        t_v6 = bench(flashnorm_v6, x, W)
        t_v5 = V5_TIMES_MS[(name, M)]
        delta = (t_real - t_v6) / t_real * 100
        sp = t_v5 / t_v6
        path = 'decode' if M <= 16 else 'prefill'
        results[(name, M)] = (t_real, t_v5, t_v6, delta, sp, path)
        print(f'{name:<15} {M:>6} {path:>8} {t_real:>11.4f} {t_v5:>10.4f} {t_v6:>10.4f} '
              f'{delta:>+11.1f}% {sp:>9.2f}x')
        del x, W
        torch.cuda.empty_cache()

## 9. Wins summary

In [ ]:
wins = [(k, v) for k, v in results.items() if v[3] >= 0]
losses = [(k, v) for k, v in results.items() if v[3] < 0]
print(f'V6 wins vs realistic:   {len(wins):>2} / 18')
print(f'V6 losses vs realistic: {len(losses):>2} / 18')
print()

print('=== all wins ===')
for (name, M), (tr, tv5, tv6, d, sp, path) in sorted(wins, key=lambda kv: -kv[1][3]):
    print(f'  {name:<15} M={M:>5} [{path:>7}]:  +{d:>5.1f}%  (r={tr:.4f} V6={tv6:.4f})')

print()
print('=== Llama decode (headline) ===')
for name in ['Llama-3.2-1B', 'Llama-3.1-8B']:
    for M in [1, 16]:
        (tr, tv5, tv6, d, sp, path) = results[(name, M)]
        v = 'WIN' if d >= 0 else 'loss'
        print(f'  {name} M={M:>2}: realistic {tr:.4f}ms, V6 {tv6:.4f}ms  [{v} {d:+.1f}%]')

print()
print('=== per-K-step analysis ===')
for name, (K, N) in SHAPES.items():
    for M in [1, 16]:
        (tr, tv5, tv6, d, sp, path) = results[(name, M)]
        if path == 'decode':
            k_steps_v6 = (K + 31) // 32
            us_per_step = tv6 * 1000 / k_steps_v6
            print(f'  {name} M={M:>2} K={K:>4}: V6 {tv6*1000:.1f}us / {k_steps_v6} steps = {us_per_step:.2f} us/step (target: ~0.25-0.35)')

## Optional: auto-disconnect

In [ ]:
import time
print('\n[auto-disconnect] V6 done. 90s.')
for i in range(9, 0, -1):
    time.sleep(10); print(f'  {i*10}s...')
try:
    from google.colab import runtime; runtime.unassign()
except Exception as e: print(f'failed: {e}')